In [67]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [68]:
df = pd.read_csv('dane/fra_perfumes.csv')
print(f"Rozmiar początkowy bazy: {df.shape}")

Rozmiar początkowy bazy: (70103, 8)


In [69]:
df.head()

,Name,Gender,Rating Value,Rating Count,Main Accords,Perfumers,Description,url
0,9am Afnanfor women,for women,3.73,174,"['citrus', 'musky', 'woody', 'aromatic', 'warm...",[],9ambyAfnanis a fragrance for women. Top notes ...,https://www.fragrantica.com/perfume/Afnan/9am-...
1,9am Dive Afnanfor women and men,for women and men,4.29,842,"['fruity', 'woody', 'green', 'warm spicy', 'ar...",[],9am DivebyAfnanis a Aromatic Aquatic fragrance...,https://www.fragrantica.com/perfume/Afnan/9am-...
2,9am pour Femme Afnanfor women,for women,4.00,68,"['fruity', 'musky', 'amber', 'citrus', 'powder...",[],9am pour FemmebyAfnanis a Amber fragrance for ...,https://www.fragrantica.com/perfume/Afnan/9am-...
3,9pm Afnanfor men,for men,4.50,"6,865","['vanilla', 'amber', 'warm spicy', 'cinnamon',...",[],9pmbyAfnanis a Amber Vanilla fragrance for men...,https://www.fragrantica.com/perfume/Afnan/9pm-...
4,9pm pour Femme Afnanfor women,for women,3.49,63,"['woody', 'aromatic', 'rose', 'fruity', 'powde...",[],9pm pour FemmebyAfnanis a Amber Floral fragran...,https://www.fragrantica.com/perfume/Afnan/9pm-...


In [70]:
df = df.dropna(subset=['Name', 'Description'])

In [71]:
df['Rating Count'] = pd.to_numeric(df['Rating Count'], errors='coerce')
df['Rating Value'] = pd.to_numeric(df['Rating Value'], errors='coerce')
df = df.dropna(subset=['Rating Count', 'Rating Value'])

In [72]:
MIN_RATINGS = 50
df_cleaned = df[df['Rating Count'] >= MIN_RATINGS].copy()
df_cleaned = df_cleaned.reset_index(drop=True)

In [73]:
print(f"Rozmiar bazy po czyszczeniu (min. {MIN_RATINGS} ocen): {df_cleaned.shape}")

Rozmiar bazy po czyszczeniu (min. 50 ocen): (21704, 8)


In [74]:
import ast 

def clean_accords(accord_string):
    try:
        accords_list = ast.literal_eval(accord_string)
        return ", ".join(accords_list)
    except:
        return "" 

In [75]:
df_cleaned['Brand'] = df_cleaned['url'].str.extract(r'/perfume/([^/]+)/')
df_cleaned['Brand'] = df_cleaned['Brand'].str.replace('-', ' ')

df_cleaned['Perfume_ID'] = df_cleaned['url'].str.extract(r'-(\d+)\.html')

df_cleaned['Description'] = df_cleaned['Description'].str.replace(r'([a-zA-Z0-9])by([A-Z])', r'\1 by \2', regex=True)
df_cleaned['Description'] = df_cleaned['Description'].str.replace(r'([a-zA-Z])is a ', r'\1 is a ', regex=True)

df_cleaned['Clean_Accords'] = df_cleaned['Main Accords'].apply(clean_accords)

df_cleaned['Semantic_Text'] = df_cleaned['Description'] + " Main accords: " + df_cleaned['Clean_Accords'] + "."

print("Przykładowe wyciągnięte Marki i ID")
print(df_cleaned[['Brand', 'Perfume_ID', 'Clean_Accords']].head(3).to_string())

print("\nTekst Semantyczny (Semantic_Text)")
print(df_cleaned['Semantic_Text'].iloc[0])

Przykładowe wyciągnięte Marki i ID
   Brand Perfume_ID                                                                              Clean_Accords
0  Afnan      70706  citrus, musky, woody, aromatic, warm spicy, lavender, mossy, fruity, earthy, white floral
1  Afnan      78611  fruity, woody, green, warm spicy, aromatic, citrus, fresh, fresh spicy, soft spicy, amber
2  Afnan      78541                         fruity, musky, amber, citrus, powdery, sweet, animalic, soft spicy

Tekst Semantyczny (Semantic_Text)
9am by Afnan is a fragrance for women. Top notes are Lemon, Mandarin Orange, Cardamom and Pink Pepper; middle notes are Lavender, Green Apple, Orange Blossom and Rose; base notes are Musk, Moss, Cedar and Patchouli. Main accords: citrus, musky, woody, aromatic, warm spicy, lavender, mossy, fruity, earthy, white floral.


In [76]:
df_cleaned.head()

,Name,Gender,Rating Value,Rating Count,Main Accords,Perfumers,Description,url,Brand,Perfume_ID,Clean_Accords,Semantic_Text
0,9am Afnanfor women,for women,3.73,174.0,"['citrus', 'musky', 'woody', 'aromatic', 'warm...",[],9am by Afnan is a fragrance for women. Top not...,https://www.fragrantica.com/perfume/Afnan/9am-...,Afnan,70706,"citrus, musky, woody, aromatic, warm spicy, la...",9am by Afnan is a fragrance for women. Top not...
1,9am Dive Afnanfor women and men,for women and men,4.29,842.0,"['fruity', 'woody', 'green', 'warm spicy', 'ar...",[],9am Dive by Afnan is a Aromatic Aquatic fragra...,https://www.fragrantica.com/perfume/Afnan/9am-...,Afnan,78611,"fruity, woody, green, warm spicy, aromatic, ci...",9am Dive by Afnan is a Aromatic Aquatic fragra...
2,9am pour Femme Afnanfor women,for women,4.00,68.0,"['fruity', 'musky', 'amber', 'citrus', 'powder...",[],9am pour Femme by Afnan is a Amber fragrance f...,https://www.fragrantica.com/perfume/Afnan/9am-...,Afnan,78541,"fruity, musky, amber, citrus, powdery, sweet, ...",9am pour Femme by Afnan is a Amber fragrance f...
3,9pm pour Femme Afnanfor women,for women,3.49,63.0,"['woody', 'aromatic', 'rose', 'fruity', 'powde...",[],9pm pour Femme by Afnan is a Amber Floral frag...,https://www.fragrantica.com/perfume/Afnan/9pm-...,Afnan,78544,"woody, aromatic, rose, fruity, powdery, violet...",9pm pour Femme by Afnan is a Amber Floral frag...
4,Black Oudh Al Haramain Perfumesfor women and men,for women and men,4.12,113.0,"['woody', 'powdery', 'musky', 'amber', 'patcho...",[],Black Oudh by Al Haramain Perfumes is a Amber ...,https://www.fragrantica.com/perfume/Al-Haramai...,Al Haramain Perfumes,36596,"woody, powdery, musky, amber, patchouli, vanil...",Black Oudh by Al Haramain Perfumes is a Amber ...


In [77]:
def extract_clean_name(row):
    full_name = str(row['Name']).strip()
    gender_val = str(row['Gender']).strip()
    brand_val = str(row['Brand']).strip()
    
    if full_name.endswith(gender_val):
        full_name = full_name[:-len(gender_val)].strip()
        
    if full_name.lower().endswith(brand_val.lower()):
        full_name = full_name[:-len(brand_val)].strip()
        
    return full_name

In [78]:
df_cleaned['Clean_Name'] = df_cleaned.apply(extract_clean_name, axis=1)

df_cleaned['Semantic_Text'] = "Perfume: " + df_cleaned['Clean_Name'] + " by " + df_cleaned['Brand'] + ". " + df_cleaned['Description'] + " Main accords: " + df_cleaned['Clean_Accords'] + "."

print("Porównanie starej nazwy i nowej, czystej nazwy")
print(df_cleaned[['Name', 'Brand', 'Gender', 'Clean_Name']].head(5).to_string())

Porównanie starej nazwy i nowej, czystej nazwy
                                               Name                 Brand             Gender      Clean_Name
0                                9am Afnanfor women                 Afnan          for women             9am
1                   9am Dive Afnanfor women and men                 Afnan  for women and men        9am Dive
2                     9am pour Femme Afnanfor women                 Afnan          for women  9am pour Femme
3                     9pm pour Femme Afnanfor women                 Afnan          for women  9pm pour Femme
4  Black Oudh Al Haramain Perfumesfor women and men  Al Haramain Perfumes  for women and men      Black Oudh


In [79]:
df_cleaned = df_cleaned.drop(columns=['Name'])
df_cleaned = df_cleaned.rename(columns={'Clean_Name': 'Name'})
new_order = [
    'Perfume_ID', 'Name', 'Brand', 'Gender', 'Rating Value', 'Rating Count', 
    'Clean_Accords', 'Description', 'Semantic_Text', 'url', 'Main Accords'
]

df_cleaned = df_cleaned[new_order]

df_cleaned.head()

,Perfume_ID,Name,Brand,Gender,Rating Value,Rating Count,Clean_Accords,Description,Semantic_Text,url,Main Accords
0,70706,9am,Afnan,for women,3.73,174.0,"citrus, musky, woody, aromatic, warm spicy, la...",9am by Afnan is a fragrance for women. Top not...,Perfume: 9am by Afnan. 9am by Afnan is a fragr...,https://www.fragrantica.com/perfume/Afnan/9am-...,"['citrus', 'musky', 'woody', 'aromatic', 'warm..."
1,78611,9am Dive,Afnan,for women and men,4.29,842.0,"fruity, woody, green, warm spicy, aromatic, ci...",9am Dive by Afnan is a Aromatic Aquatic fragra...,Perfume: 9am Dive by Afnan. 9am Dive by Afnan ...,https://www.fragrantica.com/perfume/Afnan/9am-...,"['fruity', 'woody', 'green', 'warm spicy', 'ar..."
2,78541,9am pour Femme,Afnan,for women,4.00,68.0,"fruity, musky, amber, citrus, powdery, sweet, ...",9am pour Femme by Afnan is a Amber fragrance f...,Perfume: 9am pour Femme by Afnan. 9am pour Fem...,https://www.fragrantica.com/perfume/Afnan/9am-...,"['fruity', 'musky', 'amber', 'citrus', 'powder..."
3,78544,9pm pour Femme,Afnan,for women,3.49,63.0,"woody, aromatic, rose, fruity, powdery, violet...",9pm pour Femme by Afnan is a Amber Floral frag...,Perfume: 9pm pour Femme by Afnan. 9pm pour Fem...,https://www.fragrantica.com/perfume/Afnan/9pm-...,"['woody', 'aromatic', 'rose', 'fruity', 'powde..."
4,36596,Black Oudh,Al Haramain Perfumes,for women and men,4.12,113.0,"woody, powdery, musky, amber, patchouli, vanil...",Black Oudh by Al Haramain Perfumes is a Amber ...,Perfume: Black Oudh by Al Haramain Perfumes. B...,https://www.fragrantica.com/perfume/Al-Haramai...,"['woody', 'powdery', 'musky', 'amber', 'patcho..."
